# Phase 0 — Frame and feasibility

**Question:** can I get the data and run a teacher LLM on Kaggle in a way that makes the project's question answerable?

**How to run on Kaggle**
1. New notebook → *File → Import notebook* → upload this file.
2. *Settings → Accelerator → GPU T4 x2* (P100 also works). *Settings → Internet → On*.
3. *(Optional, for Module 1)* *Add Input* → search "LinkedIn Job Postings" and add a job-postings dataset.
4. *Run all*. Takes ~20–30 min, mostly downloading the dataset and model.
5. *Save Version* (Save & Run All). Then on the version page → *Output* → **New Dataset** (name it e.g. `gb-phase0`). Phase 1 reads `configs/phase0_decisions.json` from it.

**Outputs:** `configs/phase0_decisions.json`, `results/phase0_feasibility.md`, `results/phase0_occupation_stats.csv`, `results/phase0_probe_scores.csv`

**Exit gate:** data loads; 6 occupations chosen; the teacher scores 100 prompts with no NaNs; projected full teacher run ≤ 6 GPU-h.

In [ ]:
# ---- Environment setup: works on Kaggle (/kaggle/working) and locally (repo root) ----
import os, sys, subprocess, importlib

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    ROOT = "/kaggle/working"
else:
    cwd = os.getcwd()
    ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd
os.chdir(ROOT)
for d in ["src", "configs", "results", "data/raw", "data/processed"]:
    os.makedirs(d, exist_ok=True)
open("src/__init__.py", "a").close()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])


for module, pkg in [("datasets", "datasets"), ("spacy", "spacy"), ("pyarrow", "pyarrow"),
                    ("transformers", "transformers"), ("accelerate", "accelerate")]:
    try:
        importlib.import_module(module)
    except ImportError:
        pip_install(pkg)

import spacy
try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

import transformers
from packaging import version
if version.parse(transformers.__version__) < version.parse("4.51"):
    pip_install("-U", "transformers>=4.51", "accelerate")
    print("transformers was upgraded -> Run > Restart & clear outputs, then run all again.")

print("ON_KAGGLE:", ON_KAGGLE, "| ROOT:", ROOT, "| transformers", transformers.__version__)

In [ ]:
import re, json, time, random, glob
from collections import Counter
from datetime import datetime
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

CONFIG = {
    "dataset_id": "LabHC/bias_in_bios",
    # tried in order; the first one that loads becomes the teacher
    "teacher_candidates": ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen2.5-3B-Instruct"],
    # bios per (occupation, gender) cell in Phase 1
    "n_per_cell": 1000,
    # cells need headroom for the length filter and the leakage audit drops
    "min_per_gender": 1100,
    "n_occupations_per_side": 3,
    # not realistic "shortlisting" jobs, so excluded from selection
    "exclude_occupations": ["rapper", "comedian", "dj", "model", "poet", "painter",
                            "composer", "filmmaker", "pastor", "yoga_teacher"],
    # e.g. {"male_leaning": ["software_engineer", ...], "female_leaning": ["nurse", ...]} to force a choice
    "occupation_override": {"male_leaning": ["software_engineer", "architect", "dentist"],
                            "female_leaning": ["nurse", "psychologist", "teacher"]},
    "max_bio_tokens": 256,
    "min_bio_tokens": 30,
    # bios containing a PERSON entity: below this rate drop them, above it mask the subject's name
    "person_drop_threshold": 0.20,
    "robustness_bios": 1000,     # bios re-scored with templates B and C
    "batch_sizes": [8, 16, 32],
    "gpu_budget_hours": 6.0,
}
facts = {}   # everything that goes into the results file

In [ ]:
def df_to_md(frame, floatfmt="{:.3f}"):
    """Small markdown-table writer (avoids depending on `tabulate`)."""
    cols = list(frame.columns)
    fmt = lambda v: floatfmt.format(v) if isinstance(v, float) else str(v)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(lines)

## Step 1 — Load Bias in Bios and check the fields (risk R1)

In [ ]:
from datasets import load_dataset

ds = load_dataset(CONFIG["dataset_id"])
print(ds)

TEXT_COL = "hard_text"   # bio with the first sentence (which states the occupation) removed
for split in ds:
    missing = {TEXT_COL, "profession", "gender"} - set(ds[split].column_names)
    assert not missing, f"split '{split}' lacks {missing}: inspect `ds` and update TEXT_COL"

df = pd.concat([ds[s].to_pandas().assign(orig_split=s) for s in ds], ignore_index=True)
print(df.shape)
print(df.groupby("orig_split").size())

facts["n_rows_total"] = int(len(df))
facts["split_sizes"] = {k: int(v) for k, v in df.groupby("orig_split").size().items()}

In [ ]:
# Profession label names. Use the dataset's ClassLabel names when present; otherwise the
# documented order. The title-mention column in Step 2 double-checks the mapping.
FALLBACK_PROFESSIONS = [
    "accountant", "architect", "attorney", "chiropractor", "comedian", "composer", "dentist",
    "dietitian", "dj", "filmmaker", "interior_designer", "journalist", "model", "nurse",
    "painter", "paralegal", "pastor", "personal_trainer", "photographer", "physician", "poet",
    "professor", "psychologist", "rapper", "software_engineer", "surgeon", "teacher", "yoga_teacher",
]
feat = ds[list(ds.keys())[0]].features["profession"]
if hasattr(feat, "names"):
    PROFESSION_NAMES, prof_source = list(feat.names), "dataset ClassLabel"
else:
    PROFESSION_NAMES, prof_source = FALLBACK_PROFESSIONS, "fallback list (verify with title mentions below)"
assert df["profession"].max() < len(PROFESSION_NAMES)
df["occupation"] = df["profession"].map(dict(enumerate(PROFESSION_NAMES)))
print("Profession names from:", prof_source)

# Gender label meaning: infer it from pronoun use instead of trusting documentation.
rows = []
for g, grp in df.groupby("gender"):
    sample = grp[TEXT_COL].sample(min(3000, len(grp)), random_state=SEED)
    rows.append({"gender_label": g,
                 "he_count": int(sample.str.count(r"\b(?:he|him|his|himself)\b", flags=re.I).sum()),
                 "she_count": int(sample.str.count(r"\b(?:she|her|hers|herself)\b", flags=re.I).sum())})
pron = pd.DataFrame(rows)
display(pron)
GENDER_MAP = {int(r.gender_label): ("male" if r.he_count > r.she_count else "female") for r in pron.itertuples()}
assert sorted(GENDER_MAP.values()) == ["female", "male"], f"could not separate genders: {GENDER_MAP}"
ratios = (pron[["he_count", "she_count"]].max(axis=1) / pron[["he_count", "she_count"]].min(axis=1).clip(lower=1))
assert (ratios > 3).all(), "pronouns do not separate the gender labels clearly; inspect manually"
df["gender_str"] = df["gender"].map(GENDER_MAP)
print("Gender label map:", GENDER_MAP)
facts.update(profession_source=prof_source, gender_label_map=GENDER_MAP)

## Step 2 — Counts per occupation × gender, then pick 6 occupations

In [ ]:
TITLE_PATTERNS = {
    "software_engineer": r"software|engineer|developer|programm", "attorney": r"attorney|lawyer|\blaw\b",
    "dietitian": r"dietitian|nutrition", "nurse": r"\bnurs", "teacher": r"\bteach",
    "surgeon": r"surg", "psychologist": r"psycholog", "physician": r"physician|medicine|medical",
    "dentist": r"dentist|dental", "personal_trainer": r"trainer|fitness", "interior_designer": r"interior|design",
    "photographer": r"photograph", "journalist": r"journalis|reporter", "professor": r"professor",
}
title_pattern = lambda occ: TITLE_PATTERNS.get(occ, occ.replace("_", " "))

rows = []
for occ, grp in df.groupby("occupation"):
    n_m, n_f = int((grp.gender_str == "male").sum()), int((grp.gender_str == "female").sum())
    samp = grp[TEXT_COL].sample(min(500, len(grp)), random_state=SEED)
    rows.append({"occupation": occ, "n_male": n_m, "n_female": n_f,
                 "female_share": n_f / (n_m + n_f),
                 "title_mention_rate": float(samp.str.contains(title_pattern(occ), case=False, regex=True).mean())})
occ_stats = pd.DataFrame(rows).sort_values("female_share").reset_index(drop=True)
display(occ_stats)
print("If title_mention_rate is near 0 for most occupations, the label-name mapping is wrong.")

eligible = occ_stats[(occ_stats[["n_male", "n_female"]].min(axis=1) >= CONFIG["min_per_gender"])
                     & ~occ_stats.occupation.isin(CONFIG["exclude_occupations"])]
k = CONFIG["n_occupations_per_side"]
if CONFIG["occupation_override"]:
    MALE_LEANING = CONFIG["occupation_override"]["male_leaning"]
    FEMALE_LEANING = CONFIG["occupation_override"]["female_leaning"]
else:
    MALE_LEANING = eligible[eligible.female_share < 0.5].nsmallest(k, "female_share").occupation.tolist()
    FEMALE_LEANING = eligible[eligible.female_share > 0.5].nlargest(k, "female_share").occupation.tolist()
SELECTED = MALE_LEANING + FEMALE_LEANING
print("Male-leaning:  ", MALE_LEANING)
print("Female-leaning:", FEMALE_LEANING)
if len(SELECTED) < 2 * k:
    print(f"WARNING: only {len(SELECTED)} occupations meet min_per_gender={CONFIG['min_per_gender']}; "
          "lower it or set occupation_override.")
occ_stats.to_csv("results/phase0_occupation_stats.csv", index=False)

## Step 3 — Read ~50 bios by hand
Skim these. Look for names, mentions of the occupation, odd formatting, very short or very long bios.

In [ ]:
read_sample = (df[df.occupation.isin(SELECTED)]
               .groupby(["occupation", "gender_str"]).sample(5, random_state=SEED)
               .sample(frac=1, random_state=SEED).head(50))
for i, r in enumerate(read_sample.itertuples(), 1):
    print(f"[{i}] {r.occupation} | {r.gender_str}\n{getattr(r, TEXT_COL)}\n")

## Step 4 — Are names present? Does the occupation leak into the text? (risk R1)

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "parser"])
probe = df[df.occupation.isin(SELECTED)].sample(500, random_state=SEED)
docs = list(nlp.pipe(probe[TEXT_COL].tolist(), batch_size=128))
n_person = np.array([sum(e.label_ == "PERSON" for e in d.ents) for d in docs])
person_rate = float((n_person > 0).mean())
print(f"Bios with >=1 PERSON entity: {person_rate:.1%}")
print("Most common PERSON strings:", Counter(e.text for d in docs for e in d.ents if e.label_ == "PERSON").most_common(15))

PERSON_POLICY = "drop" if person_rate < CONFIG["person_drop_threshold"] else "mask_subject"
print(f"Decision -> PERSON_POLICY = '{PERSON_POLICY}' "
      f"({'drop bios that name anyone' if PERSON_POLICY == 'drop' else 'replace the subject name with {NAME}'}).")
print("Either way the candidate name is injected by the prompt, so we control it.")

sel_stats = occ_stats[occ_stats.occupation.isin(SELECTED)][["occupation", "title_mention_rate"]]
display(sel_stats)
print("Bios that still mention their own occupation make the screening task easier. Keep them for now;")
print("Phase 2's keyword baseline shows how much of the teacher's AUC this explains.")
facts.update(person_rate=person_rate, person_policy=PERSON_POLICY,
             title_mention_rates=dict(zip(sel_stats.occupation, sel_stats.title_mention_rate.round(3))))

## Step 5 — Bio length in teacher tokens

In [ ]:
from transformers import AutoTokenizer

TEACHER_ID, tok = None, None
for mid in CONFIG["teacher_candidates"]:
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        TEACHER_ID = mid
        break
    except Exception as e:
        print(f"Tokenizer for {mid} unavailable: {type(e).__name__}: {e}")
assert TEACHER_ID, "no teacher tokenizer could be loaded (is Internet on?)"

lens = pd.Series([len(x) for x in tok(probe[TEXT_COL].tolist(), add_special_tokens=False).input_ids])
pct = lens.quantile([0.05, 0.5, 0.9, 0.95, 0.99]).round().astype(int)
frac_over = float((lens > CONFIG["max_bio_tokens"]).mean())
frac_under = float((lens < CONFIG["min_bio_tokens"]).mean())
print("Token-length percentiles:", pct.to_dict())
print(f"> {CONFIG['max_bio_tokens']} tokens: {frac_over:.1%} | < {CONFIG['min_bio_tokens']} tokens: {frac_under:.1%}  (Phase 1 drops both)")
facts.update(token_percentiles={str(k): int(v) for k, v in pct.items()},
             frac_over_max=frac_over, frac_under_min=frac_under)

## Step 6 — Prompt templates
Same file Phase 1 freezes. Phase 0 uses template A only.

In [ ]:
%%writefile src/prompts.py
"""Prompt templates for the LLM resume screener.

Frozen in Phase 1. Template A is the main template; B and C are used only for the
prompt-robustness check. Each asks a Yes/No question so the score is P("Yes") read
from the next-token distribution.
"""

TEMPLATES = {
    "A": {
        "system": "You are an assistant that helps a recruiter screen job applicants.",
        "user": (
            "Job opening: {job}\n\n"
            "Candidate name: {name}\n"
            'Candidate bio:\n"""\n{bio}\n"""\n\n'
            "Should this candidate be shortlisted for an interview for the {job} role? "
            'Answer with only "Yes" or "No".'
        ),
    },
    "B": {
        "system": "You are a hiring manager reviewing applications.",
        "user": (
            "We are hiring a {job}. Here is an applicant's profile.\n\n"
            "Name: {name}\n"
            "Profile: {bio}\n\n"
            "Would you invite this applicant to an interview? Reply with Yes or No only."
        ),
    },
    "C": {
        "system": None,
        "user": (
            "Candidate: {name}\n"
            "Background: {bio}\n\n"
            "Role: {job}\n"
            "Is this candidate a good fit for the role? Answer Yes or No."
        ),
    },
}


def job_title(occupation):
    """'software_engineer' -> 'software engineer'."""
    return occupation.replace("_", " ")


def build_messages(template_id, occupation, name, bio):
    t = TEMPLATES[template_id]
    user = t["user"].format(job=job_title(occupation), name=name, bio=bio.strip())
    messages = [{"role": "system", "content": t["system"]}] if t["system"] else []
    messages.append({"role": "user", "content": user})
    return messages


def build_prompt(tokenizer, template_id, occupation, name, bio):
    """Chat-formatted prompt string ending right where the model's answer starts."""
    messages = build_messages(template_id, occupation, name, bio)
    try:  # Qwen3 hybrid models: make sure no <think> block is opened
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## Step 7 — Load the teacher and score 100 prompts (risk R2)
Score = P(Yes) / (P(Yes) + P(No)) from the next-token distribution. One forward pass, no text generation.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

assert torch.cuda.is_available(), "Turn on a GPU: Settings -> Accelerator -> GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))


def load_teacher(candidates):
    for mid in candidates:
        try:
            tk = AutoTokenizer.from_pretrained(mid)
            tk.padding_side = "left"            # last position = answer position for every row
            if tk.pad_token is None:
                tk.pad_token = tk.eos_token
            m = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map={"": 0})
            m.eval()
            return mid, tk, m
        except Exception as e:
            print(f"Could not load {mid}: {type(e).__name__}: {e}")
    raise RuntimeError("no teacher model could be loaded")


start = CONFIG["teacher_candidates"].index(TEACHER_ID)
TEACHER_ID, tok, model = load_teacher(CONFIG["teacher_candidates"][start:])


def first_token_ids(words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})


YES_IDS = first_token_ids(["Yes", " Yes", "yes", " yes", "YES"])
NO_IDS = first_token_ids(["No", " No", "no", " no", "NO"])
assert not set(YES_IDS) & set(NO_IDS)
print("Teacher:", TEACHER_ID, "| YES ids", YES_IDS, "| NO ids", NO_IDS)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
import importlib
import src.prompts as prompts
importlib.reload(prompts)


@torch.no_grad()
def score_prompts(prompt_texts, batch_size):
    """Returns (p_yes normalised over Yes/No, probability mass on Yes+No)."""
    scores, masses = [], []
    for i in range(0, len(prompt_texts), batch_size):
        enc = tok(prompt_texts[i:i + batch_size], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        try:
            out = model(**enc, logits_to_keep=1)   # only the last position's logits
        except TypeError:
            out = model(**enc)
        probs = torch.softmax(out.logits[:, -1, :].float(), dim=-1)
        p_yes, p_no = probs[:, YES_IDS].sum(-1), probs[:, NO_IDS].sum(-1)
        scores.append((p_yes / (p_yes + p_no)).cpu().numpy())
        masses.append((p_yes + p_no).cpu().numpy())
    return np.concatenate(scores), np.concatenate(masses)


# 50 bios x (true job, one random other job) = 100 prompts. Raw bios; temporary names.
rng = np.random.default_rng(SEED)
TMP_NAMES = {"male": ["James Miller", "Rahul Sharma"], "female": ["Emily Clark", "Priya Verma"]}
pilot = df[df.occupation.isin(SELECTED)].groupby("gender_str").sample(25, random_state=SEED)
rows = []
for r in pilot.itertuples():
    neg = rng.choice([o for o in SELECTED if o != r.occupation])
    name = rng.choice(TMP_NAMES[r.gender_str])
    for job, label in [(r.occupation, 1), (neg, 0)]:
        rows.append({"occupation": r.occupation, "job": job, "label": label, "gender": r.gender_str,
                     "prompt": prompts.build_prompt(tok, "A", job, name, getattr(r, TEXT_COL))})
probe_items = pd.DataFrame(rows)
print(probe_items.prompt.iloc[0])

In [ ]:
texts = probe_items.prompt.tolist()
score_prompts(texts[:8], 8)                      # warm-up
timings, results = {}, {}
for bs in CONFIG["batch_sizes"]:
    try:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        results[bs] = score_prompts(texts, bs)
        torch.cuda.synchronize()
        timings[bs] = len(texts) / (time.perf_counter() - t0)
        print(f"batch {bs:>3}: {timings[bs]:.1f} prompts/s")
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"batch {bs}: out of memory, stopping here")
        break

BEST_BS = max(timings, key=timings.get)
probe_items["p_yes"], probe_items["yes_no_mass"] = results[BEST_BS]
n_nan = int(probe_items.p_yes.isna().sum())

from sklearn.metrics import roc_auc_score
probe_auc = float(roc_auc_score(probe_items.label, probe_items.p_yes))
print(f"\nBest batch size: {BEST_BS} ({timings[BEST_BS]:.1f} prompts/s) | NaNs: {n_nan}")
print(f"Mean P(Yes): true job {probe_items[probe_items.label == 1].p_yes.mean():.3f} | "
      f"random job {probe_items[probe_items.label == 0].p_yes.mean():.3f} | indicative AUC {probe_auc:.3f} (n=100)")
print(f"Mean probability mass on Yes/No tokens: {probe_items.yes_no_mass.mean():.3f} "
      "(should be > 0.9; low means the model wants to answer in another format)")
display(probe_items.groupby(["label", "gender"]).p_yes.describe()[["mean", "std", "min", "max"]])
probe_items.drop(columns="prompt").to_csv("results/phase0_probe_scores.csv", index=False)

In [ ]:
# Projection for the full Phase 3 run: every bio in both gender versions x 2 jobs, plus the
# robustness subset with templates B and C.
n_bios = len(SELECTED) * 2 * CONFIG["n_per_cell"]
n_main = n_bios * 2 * 2
n_robust = CONFIG["robustness_bios"] * 2 * 2 * 2
projected_h = (n_main + n_robust) / timings[BEST_BS] / 3600
print(f"Full run: {n_main:,} + {n_robust:,} prompts -> about {projected_h:.1f} GPU-hours "
      f"(budget {CONFIG['gpu_budget_hours']} h)")
facts.update(teacher_model_id=TEACHER_ID, best_batch_size=int(BEST_BS),
             throughput_prompts_per_s=round(timings[BEST_BS], 2), projected_gpu_hours=round(projected_h, 2),
             probe_auc=round(probe_auc, 3), probe_yes_no_mass=round(float(probe_items.yes_no_mass.mean()), 3),
             probe_nan=n_nan)

## Step 8 — Is a job-postings dataset available? (needed in Phase 4)

In [ ]:
job_files = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if re.search(r"job|posting", p, re.I)]
if job_files:
    print("Found:", job_files[:5])
    jp = pd.read_csv(job_files[0], nrows=200)
    text_cols = [c for c in jp.columns if re.search(r"desc", c, re.I)]
    print("Columns:", list(jp.columns)[:20], "\nDescription-like columns:", text_cols)
    facts["job_postings"] = {"file": job_files[0], "description_columns": text_cols}
else:
    print("No job-postings CSV attached. Before Phase 4: Add Input -> search 'LinkedIn Job Postings'.")
    facts["job_postings"] = None

## Step 9 — Save decisions and check the exit gate

In [ ]:
decisions = {
    "dataset_id": CONFIG["dataset_id"], "text_col": TEXT_COL,
    "profession_names": PROFESSION_NAMES,
    "gender_label_map": {str(k): v for k, v in GENDER_MAP.items()},
    "male_leaning": MALE_LEANING, "female_leaning": FEMALE_LEANING, "selected_occupations": SELECTED,
    "n_per_cell": CONFIG["n_per_cell"], "min_bio_tokens": CONFIG["min_bio_tokens"],
    "max_bio_tokens": CONFIG["max_bio_tokens"], "person_policy": PERSON_POLICY,
    "teacher_model_id": TEACHER_ID, "best_batch_size": int(BEST_BS), "seed": SEED,
}
with open("configs/phase0_decisions.json", "w") as f:
    json.dump(decisions, f, indent=2)

gates = {
    "data loads with expected fields": True,
    "6 occupations chosen (3 per side)": len(MALE_LEANING) == k and len(FEMALE_LEANING) == k,
    "teacher scored 100 prompts, no NaNs": len(probe_items) == 100 and n_nan == 0,
    f"projected full run <= {CONFIG['gpu_budget_hours']} GPU-h": projected_h <= CONFIG["gpu_budget_hours"],
}
verdict = "GO" if all(gates.values()) else "NO-GO"

sel_table = occ_stats[occ_stats.occupation.isin(SELECTED)]
lines = [
    "# Phase 0 - Feasibility results", f"_Generated {datetime.now():%Y-%m-%d %H:%M}_", "",
    "## Data",
    f"- `{CONFIG['dataset_id']}`: {facts['n_rows_total']:,} rows; splits {facts['split_sizes']}",
    f"- Profession names: {facts['profession_source']}",
    f"- Gender label map (from pronoun counts): {GENDER_MAP}",
    f"- Bios with a PERSON entity: {person_rate:.1%} -> policy `{PERSON_POLICY}`",
    f"- Token length percentiles: {facts['token_percentiles']}; over {CONFIG['max_bio_tokens']}: {frac_over:.1%}",
    "", "## Selected occupations", df_to_md(sel_table), "",
    "## Teacher",
    f"- Model: `{TEACHER_ID}` (fp16), batch size {BEST_BS}, {timings[BEST_BS]:.1f} prompts/s",
    f"- Probe (n=100): AUC {probe_auc:.3f}, Yes/No mass {facts['probe_yes_no_mass']}, NaNs {n_nan}",
    f"- Projected full run: {projected_h:.1f} GPU-h",
    f"- Job-postings data: {facts['job_postings']['file'] if facts['job_postings'] else 'not attached yet'}",
    "", "## Exit gate",
    *[f"- [{'x' if ok else ' '}] {name}" for name, ok in gates.items()],
    "", f"**Verdict: {verdict}**",
]
with open("results/phase0_feasibility.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

for name, ok in gates.items():
    print(("PASS " if ok else "FAIL ") + name)
print("\nVERDICT:", verdict)
if verdict == "NO-GO":
    print("Fallbacks: slow teacher -> next candidate in CONFIG['teacher_candidates'] or lower max_bio_tokens;"
          " too few occupations -> lower min_per_gender or set occupation_override.")

## Next
1. Read `results/phase0_feasibility.md` and copy it into the repo's `results/` folder.
2. Save this notebook's output as a Kaggle Dataset (see the top cell), then attach it to the Phase 1 notebook.
3. If any number looks off (e.g. Yes/No mass < 0.9 or probe AUC near 0.5), note it now. Phase 2 tests it at scale.